# 02 — Labels, coverage and baselines

**Goal (Day 2 of the roadmap):** check the horizon labels built by `swissdelay.features.labels`, measure their coverage, and establish how good the simplest forecasts already are.

1. Label coverage and the common subset
2. The target Δd
3. Disruption days
4. Baselines: persistence, constant offset and historical median
9. Findings and decisions

**Reminder.** One row per prediction point and horizon. At a measured departure (delay `d0`), the target stop is the first stop whose *planned* arrival is at least *h* = 15 / 30 / 60 min after the planned departure; the label is `delta_min = target delay − d0`. Persistence predicts `delta_min = 0`.

In [1]:
# 0. Setup: imports, chart style, DuckDB view `labels` with the split of each day
import duckdb
import pandas as pd
import plotly.express as px

from swissdelay import config
from swissdelay.evaluation.splits import SPLITS, split_sql
from swissdelay.viz import HORIZON_COLORS, STATUS_COLORS, use_style

use_style()
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.3f}".format)
STATUSES = list(STATUS_COLORS)

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE VIEW labels AS
SELECT *, {split_sql()} AS split
FROM read_parquet('{config.PROCESSED}/labels_*.parquet')
""")


def q(sql: str) -> pd.DataFrame:
    """Run SQL in DuckDB and return a pandas DataFrame."""
    return con.sql(sql).df()

## 1. Label coverage and the common subset

In [2]:
# 1.1 Label status by horizon (% of prediction points)
status = q("""
SELECT horizon_min, label_status, count(*) AS n
FROM labels GROUP BY ALL
""")
status["share"] = status["n"] / status.groupby("horizon_min")["n"].transform("sum")
status.pivot(index="horizon_min", columns="label_status", values="share")[STATUSES].mul(100).round(1)

label_status,ok,terminates,cancelled,not_measured
horizon_min,,,,
15,83.400,15.900,0.400,0.200
30,61.700,37.300,0.800,0.200
60,34.500,64.600,0.800,0.100


In [3]:
# 1.2 Chart: label status by horizon
fig = px.bar(status, x="share", y=status["horizon_min"].astype(str) + " min", color="label_status",
             orientation="h", color_discrete_map=STATUS_COLORS,
             category_orders={"label_status": STATUSES}, hover_data={"n": ":,"},
             title="Why prediction points have (or lack) a label")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title=None, yaxis_title=None, legend_title=None)
fig.show()

> **Finding 1.1–1.2.**
> - Labelled: **83.4 % at 15 min, 61.7 % at 30 min, 34.5 % at 60 min**.
> - Missing labels are almost entirely **structural**: the train terminates before the horizon (15.9 / 37.3 / 64.6 %). Data problems are marginal: cancelled targets 0.4–0.8 %, unmeasured targets 0.1–0.2 %.
> - So label coverage reflects journey length, not data quality, and dropping unlabelled points does not hide disrupted cases in a significant way (cancellations: < 1 %).

In [4]:
# 1.3 Share labelled by category and horizon (%)
q("""
SELECT category, horizon_min, avg((label_status = 'ok')::INT) AS share_labelled
FROM labels GROUP BY ALL
""").pivot(index="category", columns="horizon_min", values="share_labelled").mul(100).round(1)

horizon_min,15,30,60
category,,,
EC,84.600,69.600,48.400
IC,90.000,79.000,58.300
IR,90.300,69.700,43.900
RE,74.600,47.900,16.900


> **Finding 1.3.**
> - Coverage follows journey length: IC 90 / 79 / 58 %, EC 85 / 70 / 48 %, IR 90 / 70 / 44 %, **RE 75 / 48 / 17 %** at 15 / 30 / 60 min.
> - RE journeys are short: at 60 min, only one RE point in six has a target stop.

In [5]:
# 1.4 Share labelled by split and horizon (%): is coverage stable over time?
cov_split = q("""
SELECT split, horizon_min, avg((label_status = 'ok')::INT) AS share_labelled
FROM labels GROUP BY ALL
""").pivot(index="split", columns="horizon_min", values="share_labelled").reindex(list(SPLITS))
cov_split.mul(100).round(1)

horizon_min,15,30,60
split,,,
train,83.400,61.900,34.600
valid,83.400,61.500,34.400
test,83.300,61.500,34.500
production,83.300,61.300,34.100


> **Finding 1.4.** Coverage is **stable across splits** (within 0.5 point at every horizon): train, validation, test and production are labelled in the same proportions, so differences in model error between splits will not come from label coverage.

In [6]:
# 1.5 Size of the common subset (labelled at 15, 30 and 60 min) by split and category
common = q("""
SELECT split, category,
       count(*) FILTER (WHERE horizon_min = 15)                       AS points,
       count(*) FILTER (WHERE horizon_min = 15 AND in_common_subset)  AS common_points
FROM labels GROUP BY ALL  -- one row per point and horizon: count each point once
""")
common["share_common"] = common["common_points"] / common["points"]
display(common.groupby("split")[["points", "common_points"]].sum().reindex(list(SPLITS))
        .assign(share_common=lambda d: d["common_points"] / d["points"]))
common.pivot(index="category", columns="split", values="share_common")[list(SPLITS)].mul(100).round(1)

,points,common_points,share_common
split,,,
train,4776986,1648454,0.345
valid,531549,182706,0.344
test,511627,175420,0.343
production,1559631,531591,0.341


split,train,valid,test,production
category,,,,
EC,47.900,49.700,48.300,48.000
IC,57.600,57.000,60.000,59.100
IR,44.100,43.200,43.600,43.300
RE,17.400,16.500,15.500,15.700


> **Finding 1.5.**
> - The common subset (labelled at 15, 30 and 60 min) holds **34 % of prediction points**: train 1.65 M, validation 183 k, test 175 k, production 532 k. Ample for every model.
> - Its composition differs by category: 58–60 % of IC points, 48–50 % of EC, 43–44 % of IR, but only **16–17 % of RE points**.

In [7]:
# 1.6 Is the common subset biased? Current delay inside vs outside it (one row per point)
q("""
WITH p AS (
  SELECT DISTINCT trip_key, stop_seq, category, d0_min, in_common_subset FROM labels
)
SELECT in_common_subset, count(*) AS points,
       avg(d0_min) AS mean_d0,
       quantile_cont(d0_min, 0.5) AS median_d0,
       avg((d0_min > 3)::INT) AS share_d0_over_3min,
       avg((category IN ('IC', 'EC'))::INT) AS share_ic_ec
FROM p GROUP BY ALL
""")

,in_common_subset,points,mean_d0,median_d0,share_d0_over_3min,share_ic_ec
0,True,2538171,1.401,0.833,0.090,0.270
1,False,4841622,1.383,0.783,0.088,0.107


> **Finding 1.6.**
> - The common subset is **not biased on the current delay**: mean d0 1.40 vs 1.38 min, median 0.83 vs 0.78, share over 3 min 9.0 vs 8.8 % (inside vs outside).
> - It **is biased on train type**: IC + EC are 27 % of the common subset vs 11 % outside, because it keeps long journeys.
>
> **Decision:** the headline table stays on the common subset (same points at every horizon), and results are **also reported per category** and on each horizon's full labelled population, so RE is not under-represented in the conclusions.

In [8]:
# 1.7 How far beyond h is the target stop? Scheduled gap (min) for labelled points
q("""
SELECT horizon_min,
       quantile_cont(sched_gap_min, [0.5, 0.9, 0.99]) AS gap_p50_p90_p99,
       avg(sched_gap_min - horizon_min) AS mean_overshoot
FROM labels WHERE label_status = 'ok'
GROUP BY ALL ORDER BY horizon_min
""")

,horizon_min,gap_p50_p90_p99,mean_overshoot
0,15,"[19.0, 30.0, 56.0]",6.800
1,30,"[35.0, 48.0, 74.0]",7.569
2,60,"[66.0, 78.0, 114.0]",8.440


> **Finding 1.7.**
> - The target stop is on average **7–8 min beyond the nominal horizon**: median scheduled gap 19 min (h = 15), 35 min (h = 30), 66 min (h = 60); 90 % of targets within 30 / 48 / 78 min.
> - "15 min ahead" therefore means "the first stop at least 15 min ahead", which is typically ~20 min.
>
> **Decision:** the scheduled gap `sched_gap_min` is known at prediction time (timetable only) and becomes a **model feature**; results are reported per nominal horizon.

## 2. The target Δd

`delta_min` = delay at the target stop − current delay. Positive: the train loses time; negative: it recovers. The mean absolute Δd is exactly the error of persistence.

In [9]:
# 2.1 Distribution of Δd by horizon (labelled points, all splits)
q("""
SELECT horizon_min, count(*) AS n,
       avg(delta_min) AS mean,
       quantile_cont(delta_min, 0.01) AS p01,
       quantile_cont(delta_min, 0.10) AS p10,
       quantile_cont(delta_min, 0.50) AS p50,
       quantile_cont(delta_min, 0.90) AS p90,
       quantile_cont(delta_min, 0.99) AS p99,
       avg(abs(delta_min)) AS mean_abs,
       avg((abs(delta_min) < 1)::INT) AS share_within_1min,
       avg((delta_min > 3)::INT) AS share_loses_over_3min,
       avg((delta_min < -3)::INT) AS share_recovers_over_3min
FROM labels WHERE label_status = 'ok'
GROUP BY ALL ORDER BY horizon_min
""")

,horizon_min,n,mean,p01,p10,p50,p90,p99,mean_abs,share_within_1min,share_loses_over_3min,share_recovers_over_3min
0,15,6154352,-0.784,-4.283,-2.400,-0.850,0.750,3.867,1.297,0.477,0.016,0.046
1,30,4555263,-0.748,-5.167,-2.650,-0.817,1.017,5.133,1.450,0.455,0.025,0.071
2,60,2545031,-0.716,-6.400,-2.900,-0.800,1.300,6.900,1.641,0.432,0.036,0.092


In [10]:
# 2.2 Chart: distribution of Δd by horizon (clipped to [-10, 15] min)
hist = q("""
SELECT horizon_min, greatest(least(floor(delta_min), 15), -10) AS minute, count(*) AS n
FROM labels WHERE label_status = 'ok'
GROUP BY ALL ORDER BY minute
""")
hist["share"] = hist["n"] / hist.groupby("horizon_min")["n"].transform("sum")
hist["horizon"] = hist["horizon_min"].astype(str) + " min"
fig = px.bar(hist, x="minute", y="share", color="horizon", barmode="group",
             color_discrete_map={f"{h} min": c for h, c in HORIZON_COLORS.items()},
             title="Change in delay Δd by horizon (clipped to [-10, 15] min)")
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="Δd (min, floored)", yaxis_title=None,
                  legend_title=None)
fig.show()

> **Finding 2.1–2.2.**
> - Δd is concentrated around zero but **shifted below it**: median −0.85 / −0.82 / −0.80 min, mean −0.78 / −0.75 / −0.72 min at 15 / 30 / 60 min.
> - Spread grows slowly with the horizon: 10–90 % range [−2.4, +0.75] at 15 min, [−2.9, +1.3] at 60 min; 1–99 % range [−4.3, +3.9] to [−6.4, +6.9].
> - 48 / 46 / 43 % of points change by less than 1 min. Losing more than 3 min is rare (1.6 / 2.5 / 3.6 %), recovering more than 3 min more frequent (4.6 / 7.1 / 9.2 %).
> - **Why the negative shift:** the label compares a *departure* delay now with an *arrival* delay at the target. Swiss timetables include running-time reserves before stations, so trains typically arrive ≈ 1 min before the planned arrival while departing on time (see the IC51 example in notebook 01, Finding 7.6). Part of Δd is therefore a **structural offset of about −0.8 min**, not a real recovery.

In [11]:
# 2.3 Δd depends on the current delay: late trains tend to recover (labelled points, train split)
q("""
SELECT horizon_min,
       CASE WHEN d0_min < 1 THEN 'a: < 1' WHEN d0_min < 3 THEN 'b: 1-3'
            WHEN d0_min < 10 THEN 'c: 3-10' ELSE 'd: >= 10' END AS d0_bucket,
       count(*) AS n, avg(delta_min) AS mean_delta, quantile_cont(delta_min, 0.5) AS median_delta,
       avg(abs(delta_min)) AS mean_abs_delta
FROM labels WHERE label_status = 'ok' AND split = 'train'
GROUP BY ALL ORDER BY horizon_min, d0_bucket
""")

,horizon_min,d0_bucket,n,mean_delta,median_delta,mean_abs_delta
0,15,a: < 1,2353419,-0.465,-0.550,1.022
1,15,b: 1-3,1299578,-1.138,-1.250,1.493
2,15,c: 3-10,300019,-1.853,-1.967,2.263
3,15,d: >= 10,32277,-0.558,-1.317,3.077
4,30,a: < 1,1748182,-0.307,-0.467,1.059
5,30,b: 1-3,961427,-1.149,-1.333,1.642
6,30,c: 3-10,223667,-2.550,-2.833,3.065
7,30,d: >= 10,23986,-1.029,-2.150,4.632
8,60,a: < 1,966328,-0.163,-0.417,1.147
9,60,b: 1-3,546437,-1.108,-1.383,1.773


> **Finding 2.3.**
> - Late trains recover, and the more they are late, the more they recover (regression to the mean): median Δd for a current delay of 3–10 min is −2.0 / −2.8 / −3.5 min at 15 / 30 / 60 min, vs −0.55 / −0.47 / −0.42 for trains on time.
> - Even trains on time (< 1 min) have a negative Δd: the structural offset of 2.1–2.2.
> - Uncertainty is concentrated in the **already late trains**: mean |Δd| 1.0–1.1 min for d0 < 1 min, but 3.1 / 4.6 / 6.9 min for d0 ≥ 10 min. This is where models can make a difference.

In [12]:
# 2.4 First baseline preview: persistence error (mean |Δd|) on the common subset, by split
q("""
SELECT split, horizon_min, count(*) AS n,
       avg(abs(delta_min)) AS persistence_mae,
       quantile_cont(abs(delta_min), 0.9) AS persistence_p90
FROM labels WHERE in_common_subset
GROUP BY ALL ORDER BY horizon_min, split
""").pivot(index="split", columns="horizon_min", values=["persistence_mae", "persistence_p90"]).reindex(list(SPLITS))

persistence_mae             persistence_p90            
horizon_min              15    30    60              15    30    60
split                                                              
train                 1.310 1.442 1.602           2.567 2.917 3.300
valid                 1.325 1.474 1.637           2.600 2.967 3.367
test                  1.399 1.581 1.797           2.750 3.217 3.817
production            1.340 1.508 1.703           2.633 3.050 3.533

> **Finding 2.4 (first benchmark result).** Persistence (Δd = 0) on the common subset:
>
> | Split | MAE 15 / 30 / 60 min | P90 abs. error 15 / 30 / 60 min |
> |---|---|---|
> | Train | 1.31 / 1.44 / 1.60 | 2.57 / 2.92 / 3.30 |
> | Validation | 1.33 / 1.47 / 1.64 | 2.60 / 2.97 / 3.37 |
> | Test | 1.40 / 1.58 / 1.80 | 2.75 / 3.22 / 3.82 |
> | Production | 1.34 / 1.51 / 1.70 | 2.63 / 3.05 / 3.53 |
>
> - Errors are small in absolute terms (≈ 1.3–1.8 min) and grow slowly with the horizon.
> - **The test month is the hardest** (+7 % MAE vs train at 15 min, +12 % at 60 min), consistent with June being more disrupted (notebook 01, Finding 5.2–5.4).
> - Part of persistence's error is the structural offset of −0.8 min (2.1–2.2), which any model learns trivially. A **constant-offset baseline** separates that trivial gain from real skill.

## 3. Disruption days

Built by `swissdelay.models.baselines` (or `swissdelay.evaluation.days`): for each day, the mean departure delay and the share of runs with a cancelled stop, combined as the average of their z-scores. The **worst 5 % of days (22 of 426)** are disruption days. It is a reporting stratum, not a model feature.

In [ ]:
# 3.1 The disruption days
from swissdelay.evaluation.days import DAILY_STATS_PATH

daily = pd.read_parquet(DAILY_STATS_PATH)
cols = ["operating_day", "weekday", "split", "mean_delay_min", "p90_delay_min",
        "cancelled_runs_share", "disruption_score"]
daily[daily["is_disruption_day"]].sort_values("disruption_score", ascending=False)[cols]

In [ ]:
# 3.2 Chart: daily mean delay vs share of runs with a cancellation
daily["day_type"] = daily["is_disruption_day"].map({True: "disruption day", False: "normal day"})
fig = px.scatter(daily, x="cancelled_runs_share", y="mean_delay_min", color="day_type",
                 color_discrete_map={"disruption day": "#e34948", "normal day": "#8a8984"},
                 hover_data=["operating_day", "weekday", "split"],
                 title="Each day of the study period: delay vs cancellations")
fig.update_traces(marker=dict(size=8, line=dict(width=1, color="#fcfcfb")))
fig.update_layout(xaxis_tickformat=".0%", xaxis_title="runs with a cancelled stop",
                  yaxis_title="mean departure delay (min)", legend_title=None)
fig.show()

In [ ]:
# 3.3 Disruption days by split and by day of week
display(daily.groupby("split")["is_disruption_day"].agg(["sum", "count", "mean"]).reindex(list(SPLITS)))
daily[daily["is_disruption_day"]]["weekday"].value_counts()

> **Finding 3.1–3.3.**
> - 22 disruption days. The worst is **Monday 16 Feb 2026** (20 % of runs with a cancelled stop, mean delay 2.0 min), followed by 17 Feb: the incident already seen in notebook 01. Then a series of **summer 2026 weekdays** with high delays (26 and 19 June: mean delay 2.0–2.3 min, P90 4–5 min).
> - Combining delay with cancellations shifts the picture: weekend engineering works have many cancellations but low delays, so weekdays now dominate (Thursday 5, Monday 4, Friday 4; weekends 5 of 22).
> - **They are very unevenly spread:** train 3 of 273 days (1 %), **validation 0 of 31**, test 9 of 30 (30 %), production 10 of 92 (11 %). June–August 2026 is far more disrupted than the rest of the period.
> - **Problem:** with a single threshold over all days, the validation month has no disruption day, so the normal-vs-disruption breakdown cannot be used for model selection, and the training data contains almost none.

## 4. Baselines

Fitted on the training months only (`swissdelay.models.baselines`):

- **persistence:** Δd = 0.
- **offset:** Δd = training median per horizon (≈ −0.8 min). **The reference to beat.**
- **historical:** training median Δd by line, station, hour and day type, with fallbacks to coarser groups (≥ 30 labels).

MAE in minutes with 95 % day-bootstrap confidence intervals. `diff_vs_ref` = MAE − MAE(offset) on the same points (negative = better than the offset); significant when `[diff_lo, diff_hi]` excludes 0.

In [ ]:
# 4.1 Validation month, common subset: MAE by horizon and model
from swissdelay.evaluation.metrics import evaluate
from swissdelay.models.baselines import PRED_COLS, PREDICTIONS_PATH

def preds(where: str) -> pd.DataFrame:
    return q(f"SELECT * FROM read_parquet('{PREDICTIONS_PATH}') WHERE {where}")

valid = preds("split = 'valid' AND in_common_subset")
res_valid = evaluate(valid, PRED_COLS, reference="pred_offset")
res_valid.round(3)

In [ ]:
# 4.2 Chart: MAE by horizon and model, validation month (95 % CI)
r = res_valid.copy()
r["err_plus"], r["err_minus"] = r["mae_hi"] - r["mae"], r["mae"] - r["mae_lo"]
fig = px.bar(r, x="horizon_min", y="mae", color="model", barmode="group",
             error_y="err_plus", error_y_minus="err_minus",
             color_discrete_map={"persistence": "#8a8984", "offset": "#2a78d6", "historical": "#eb6834"},
             title="Baselines on the validation month (common subset)")
fig.update_layout(xaxis_title="horizon (min)", yaxis_title="MAE (min)", legend_title=None,
                  xaxis=dict(type="category"))
fig.show()

> **Finding 4.1–4.2 (validation, common subset).**
>
> | Horizon | Persistence | Offset | **Historical** | Historical − offset [95 % CI] |
> |---|---|---|---|---|
> | 15 min | 1.33 | 1.06 | **0.88** | −0.18 [−0.18, −0.17] |
> | 30 min | 1.47 | 1.25 | **1.07** | −0.18 [−0.18, −0.17] |
> | 60 min | 1.64 | 1.44 | **1.28** | −0.16 [−0.17, −0.16] |
>
> - The constant offset alone removes 20 / 15 / 12 % of persistence's error: that part is structural, not skill.
> - The historical median (line × station × hour × day type) removes a further **17 / 14 / 11 %** vs the offset, with intervals far from 0: where and when a train runs carries real information (station-specific running-time reserves). Overall −33 / −27 / −22 % vs persistence.
> - The historical baseline does **not** use the current delay.

In [ ]:
# 4.3 Breakdowns on the validation month: by category, by current delay, normal vs disruption days
valid["d0_bucket"] = pd.cut(valid["d0_min"], [-99, 1, 3, 10, 999],
                            labels=["on time < 1", "minor 1-3", "moderate 3-10", "major > 10"], right=False)
for by in ["category", "d0_bucket", "is_disruption_day"]:
    res = evaluate(valid, PRED_COLS, group_cols=("horizon_min", by), reference="pred_offset", n_boot=300)
    display(res.pivot_table(index=by, columns=["horizon_min", "model"], values="mae", observed=True).round(2))

> **Finding 4.3 (validation breakdowns, MAE in min at 15 / 30 / 60 min).**
> - **By category:** EC is the hardest (historical 1.46 / 2.14 / 2.44), IR the easiest (0.80 / 0.97 / 1.16). The historical gain over the offset is largest for **RE (−22 % at 15 min)** and smallest for IC (−12 %).
> - **By current delay:** error grows steeply with the delay already accumulated: historical 0.78 / 0.87 / 1.01 for trains on time, 1.40 / 2.20 / 2.87 for 3–10 min, **2.92 / 4.49 / 6.27 for > 10 min**. For late trains the historical gain is small (−8 to −12 % vs offset at > 10 min): it ignores the current delay, while notebook section 2.3 shows late trains recover in proportion to it. **This is the first signal for Day 3.**
> - **Normal vs disruption days:** not measurable on validation (no disruption day, see 3.3).

In [ ]:
# 4.4 Historical baseline: which fallback level answered? (% of validation rows)
q(f"""
SELECT horizon_min, hist_level, count(*) AS n
FROM read_parquet('{PREDICTIONS_PATH}') WHERE split = 'valid'
GROUP BY ALL ORDER BY horizon_min, hist_level
""").pivot(index="hist_level", columns="horizon_min", values="n").pipe(lambda d: d / d.sum() * 100).round(1)

> **Finding 4.4.** The historical groups are well populated: **97 / 97 / 95 %** of validation rows are answered at the finest level (line × station × hour × day type), 2–4 % at line × station × hour, and less than 1 % further down. The fallbacks matter little.

In [ ]:
# 4.5 Each horizon's full labelled population (not only the common subset), validation month
full = preds("split = 'valid'")
evaluate(full, PRED_COLS, reference="pred_offset").round(3)

> **Finding 4.5.** On each horizon's full labelled population (443 k / 327 k / 183 k points), the results are the same: historical 0.85 / 1.06 / 1.28 min, offset 1.04 / 1.23 / 1.45, persistence 1.29 / 1.45 / 1.64. The common-subset restriction does not change the conclusions.

In [ ]:
# 4.6 Test month, common subset (look once, at the end of Day 2)
test = preds("split = 'test' AND in_common_subset")
evaluate(test, PRED_COLS, reference="pred_offset").round(3)

> **Finding 4.6 (test month, looked at once).** Errors are ≈ 10–15 % higher than on validation (historical 0.99 / 1.23 / 1.49 min; offset 1.16 / 1.38 / 1.65; persistence 1.40 / 1.58 / 1.80), consistent with June being disrupted (9 disruption days). The ranking and the gains are unchanged (historical − offset = −0.16 / −0.15 / −0.16, CI excludes 0). From now on models are compared on validation only; no tuning on test.

## 9. Findings and decisions

### What Day 2 established

- **Labels** (`swissdelay.features.labels`): at each measured departure, the target stop is the first stop whose *planned* arrival is ≥ *h* after the planned departure (timetable only, leakage-tested), and Δd = arrival delay at the target − current delay.
- **Coverage:** 83.4 / 61.7 / 34.5 % labelled at 15 / 30 / 60 min, stable across splits. Missing labels are structural (the train terminates before *h*); cancelled or unmeasured targets are < 1 %. Coverage follows journey length (RE: 17 % at 60 min; IC: 58 %).
- **Common subset** (labelled at all three horizons): 34 % of points — train 1.65 M, validation 183 k, test 175 k, production 532 k. Same current-delay distribution as outside, but more IC / EC (27 % vs 11 %).
- **Effective horizon:** the target is 7–8 min beyond *h* on average (median 19 / 35 / 66 min).
- **Δd** is shifted by **≈ −0.8 min** at every horizon: departure delay now vs arrival delay at the target, trains arriving slightly early. Late trains recover in proportion to their delay; uncertainty is concentrated in already-late trains.
- **Baselines** (validation, common subset, MAE in min at 15 / 30 / 60): persistence 1.33 / 1.47 / 1.64, constant offset 1.06 / 1.25 / 1.44, **historical median 0.88 / 1.07 / 1.28**. Historical beats the offset significantly at every horizon (−0.16 to −0.18 min, CI excludes 0) and on test (0.99 / 1.23 / 1.49).
- **Where errors are:** EC and late trains (> 10 min late: 2.9 / 4.5 / 6.3 min); the historical baseline ignores the current delay, which is where Day 3 should gain.

### Decisions

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Prediction time | *t* = measured departure; the target stop is chosen from the timetable only | labels tests |
| 2 | Never-target stops | Skipped when choosing the target (pass-through, invalid station, inconsistent planned times); cancelled targets are not skipped (status `cancelled`) | 1.1 |
| 3 | Headline population | Common subset, plus per-category and per-horizon full populations | 1.5–1.6, 4.5 |
| 4 | Scheduled gap | `sched_gap_min` is known in advance and becomes a model feature from Day 3 | 1.7 |
| 5 | Reference baseline | The **constant offset** (training median Δd per horizon) is the reference; a model shows skill only if it beats it | 2.1–2.4 |
| 6 | Bar for Day 3 | The **historical median** baseline: 0.88 / 1.07 / 1.28 min on validation | 4.1–4.2 |
| 7 | Test month | Seen once for the baselines; models are compared on validation only | 4.6 |

### Open decision

- **Disruption-day definition (3.3).** A single top-5 % threshold over all days puts 0 disruption days in validation and 3 in training, because summer 2026 is much more disrupted. Proposed fix: compute the z-scores and the threshold (95th percentile) **on training days only**, then apply them to every day: a disruption day becomes "a day worse than 95 % of training days". About 14 training days, and validation and test get as many as they really have, with nothing learned from future days.

### Next (Day 3)

Ridge and train-only XGBoost with the current delay, the recent trajectory (last 1 / 3 / 5 stops), timetable slack and `sched_gap_min`; history and slack ablations; compared to the historical baseline on validation.